In [11]:
# Instalando dependencias para lidar com arquivos parquet.
!pip install -q pyarrow fastparquet

In [12]:
import pandas as pd
import pyarrow, fastparquet

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
# Caminho base para os arquivos da camada gold
BASE_GOLD   = "/content/drive/MyDrive/projeto-medalhao/gold/"

In [ ]:
# Método para incrementar a tabela resultante da camada gold 'ml_aluno' com dados socioeconomicos,
# buscando caracterizar melhor municípios e UFs brasileiras.
def constroi_df_resultante(df, df_estatisticas_escolares, df_dados_socioeconomicos):

  df_dados_socioeconomicos.loc[df_dados_socioeconomicos['gini_uf'] > 1, 'gini_uf'] /= 1000

  df_estatisticas_escolares = df_estatisticas_escolares \
  .filter(items=["ano", "id_escola", "total_alunos", "percentual_faltantes", "desvio_padrao_proficiencia"]) \
  .rename(columns={
      "total_alunos": "total_participantes_escola",
      "percentual_faltantes": "percentual_faltantes_escola",
      "desvio_padrao_proficiencia": "desvio_padrao_proficiencia_escola"})

  df = df.merge(df_estatisticas_escolares, on=["id_escola", "ano"], how="left")

  df['nome_municipio'] = df['nome_municipio'].str.upper()
  df_dados_socioeconomicos = df_dados_socioeconomicos.rename(columns={"municipio": "nome_municipio"})

  df_dados_socioeconomicos['nome_municipio'] = df_dados_socioeconomicos['nome_municipio'].replace('´', "'", regex=True)

  df = df.merge(df_dados_socioeconomicos, on=["nome_municipio", "sigla_uf", "ano"], how="left")

  return df.drop(columns=["_gold_processed_at"])

# Encontramos alguns valores faltantes para média de português municipal e estadual, que não haviam sido percebidos durante a construção do pipeline de dados.

###**Estadual**:
    Foram encontrados valores faltantes para sigla_uf = ("Não encontrado", "DF", "TO")
      
        Não encontrado (489 valores): não houve match entre o id do município reportado para aquela escola e o mapeamento de ids municipais fornecido.
        Decisão - descartar.
        
        DF (22111 valores): ao consultar a tabela de dados brutos estaduais do Tech Challenge 2, percebemos que ela não continha dados para o Distrito Federal.
        Decisão - utilizar a média do único município que compõe o estado (Brasília).

        TO (24 valores): uma investigação mais detalhada no pipeline é necessária para entender o motivo pelo qual registros da rede privada da cidade de Gurupi, em 2024, ficaram com valores nulos para média municipal e estadual.
        Decisão - popular com o valor conhecido de média estadual para TO em 2024 (742.86).

###**Municipal**
    Para os municípios, a estratégia foi atribuir aos registros faltantes o valor da média das outras redes de ensino naquele mesmo ano.
    Quando não disponível, foi atribuída a média estadual naquele ano.

In [15]:
def imputa_faltantes(df):
  # Descartando não encontrados.
  df = df[df['sigla_uf'] != "Não encontrado"]

  # Atribuindo o valor conhecido de média estadual para TO em 2024.
  MEDIA_TO_2024 = 742.86
  df.loc[df['feat_media_portugues_estado'].isna() & (df['sigla_uf'] == 'TO'), 'feat_media_portugues_estado'] = MEDIA_TO_2024

  # Atribuindo a média ao Distrito Federal.
  MEDIA_BRASILIA_2024 = 743.01
  df.loc[(df['sigla_uf'] == 'DF') & (df['ano'] == 2024), 'feat_media_portugues_estado'] = MEDIA_BRASILIA_2024

  # Olhando para a media municipal.
  mun_medias_faltantes = df[df['feat_media_portugues_municipio'].isna() == True]["nome_municipio"].unique().tolist()

  for municipio in mun_medias_faltantes:
    for ano in [2023, 2024]:

      total_municipais_nulas = df.loc[(df['nome_municipio'] == municipio) & (df['ano'] == ano), 'feat_media_portugues_municipio'].isna().sum()

      # checa primeiro se existe algum registro não nulo para média do município naquele ano.
      # caso todos sejam nulos, é necessario atribuir a média estadual.
      if df.loc[(df['nome_municipio'] == municipio) & (df['ano'] == ano), 'feat_media_portugues_municipio'].count() == 0:
        df.loc[(df['nome_municipio'] == municipio) & (df['ano'] == ano), 'feat_media_portugues_municipio'] = \
          df.loc[(df['nome_municipio'] == municipio) & (df['ano'] == ano), 'feat_media_portugues_estado'].unique().mean()

      elif total_municipais_nulas > 0:

        df.loc[
            (df['feat_media_portugues_municipio'].isna() == True) & (df['nome_municipio'] == municipio) & (df['ano'] == ano), 'feat_media_portugues_municipio'] = \
            df.loc[(df['feat_media_portugues_municipio'].isna() == False) & (df['nome_municipio'] == municipio) & \
            (df['ano'] == ano), 'feat_media_portugues_municipio'].unique().mean()

  return df

In [16]:
def preprocessing(df_ml_aluno, df_estatisticas_escolares, df_dados_socioeconomicos):

  df_ml_aluno = imputa_faltantes(df_ml_aluno)

  df_resultante = constroi_df_resultante(df_ml_aluno, df_estatisticas_escolares, df_dados_socioeconomicos)

  df_resultante = df_resultante.filter(items=['ano', 'feat_rede_encoded', 'feat_peso_aluno', 'feat_media_proficiencia_escola', 'desvio_padrao_proficiencia_escola', \
                                            'total_participantes_escola', 'percentual_faltantes_escola', 'feat_media_portugues_municipio', \
                                            'feat_media_portugues_estado', 'proporcao_pbf_municipio', 'ideb_medio_municipio', \
                                            'tendencia_ideb_municipio', 'gini_uf', 'target_alfabetizado']) \
                              .rename(columns={'desvio_padrao_proficiencia_escola': 'feat_std_proficiencia_escola', 'total_participantes_escola': 'feat_total_participantes_escola', \
                                            'percentual_faltantes_escola': 'feat_percentual_faltantes_escola', 'proporcao_pbf_municipio': 'feat_proporcao_pbf_municipio', \
                                            'ideb_medio_municipio': 'feat_ideb_medio_municipio', 'tendencia_ideb_municipio': 'feat_tendencia_ideb_municipio', \
                                            'gini_uf': 'feat_gini_uf'})

  return df_resultante

In [ ]:
df_ml_aluno = pd.read_parquet(f'{BASE_GOLD}/ml_aluno', engine='pyarrow')
df_estatisticas_escolares = pd.read_parquet(f'{BASE_GOLD}/estatisticas_escolares', engine='pyarrow')
df_dados_socioeconomicos = pd.read_csv("/content/br_dados_socioeconomicos.csv")

df_resultante = preprocessing(df_ml_aluno, df_estatisticas_escolares, df_dados_socioeconomicos)
df_resultante.describe()

,ano,feat_rede_encoded,feat_peso_aluno,feat_media_proficiencia_escola,feat_std_proficiencia_escola,feat_total_participantes_escola,feat_percentual_faltantes_escola,feat_media_portugues_municipio,feat_media_portugues_estado,feat_proporcao_pbf_municipio,feat_ideb_medio_municipio,feat_tendencia_ideb_municipio,feat_gini_uf,target_alfabetizado
count,3.354172e+06,3.354172e+06,3.354172e+06,3.354172e+06,3.354172e+06,3.354172e+06,3.354172e+06,3.354172e+06,3.354172e+06,3.354172e+06,3.354172e+06,3.354172e+06,3.354172e+06,3.354172e+06
mean,2.023552e+03,1.111085e+00,1.148532e+00,7.483780e+02,4.038280e+01,7.695843e+01,1.182717e+01,7.480108e+02,7.477471e+02,2.882226e-01,5.684785e+00,-9.115644e-02,4.263845e+02,5.915684e-01
std,4.972848e-01,3.143056e-01,3.585416e-01,2.465181e+01,7.980302e+00,5.050947e+01,9.015035e+00,2.030736e+01,1.552018e+01,1.757630e-01,7.750341e-01,4.969347e-01,1.569276e+02,4.915438e-01
min,2.023000e+03,1.000000e+00,1.000000e-01,6.010600e+02,0.000000e+00,3.000000e+00,0.000000e+00,6.733000e+02,7.125600e+02,1.000000e-02,3.170000e+00,-2.300000e+00,4.300000e-01,0.000000e+00
25%,2.023000e+03,1.000000e+00,1.000000e+00,7.328700e+02,3.539000e+01,4.100000e+01,5.330000e+00,7.358000e+02,7.373000e+02,1.500000e-01,5.130000e+00,-4.000000e-01,4.580000e+02,0.000000e+00
50%,2.024000e+03,1.000000e+00,1.090000e+00,7.469500e+02,4.065000e+01,6.600000e+01,1.028000e+01,7.461800e+02,7.472800e+02,2.400000e-01,5.810000e+00,-1.000000e-01,4.770000e+02,1.000000e+00
75%,2.024000e+03,1.000000e+00,1.210000e+00,7.614900e+02,4.560000e+01,1.010000e+02,1.667000e+01,7.568300e+02,7.546800e+02,4.000000e-01,6.300000e+00,2.000000e-01,4.890000e+02,1.000000e+00
max,2.024000e+03,4.000000e+00,1.425500e+02,8.850200e+02,1.179100e+02,4.410000e+02,9.895000e+01,8.684600e+02,7.973400e+02,1.000000e+00,8.730000e+00,3.400000e+00,5.610000e+02,1.000000e+00
